In [0]:
# ============================================================
# Silver — Source 09: SFTP Supplier Catalog
#
# Transformations:
#   - Cast all numeric strings to correct types
#   - Cast updated_date string to date
#   - Validate unit_cost_gbp > 0
#   - Validate unit_cost_gbp < rrp_gbp (supplier cost < retail)
#   - Reject null product_sku → quarantine
#   - Deduplicate on product_sku — keep latest updated_date
#
# Source:  bronze.src_09_suppliers.supplier_catalog
# Target:  silver.src_09_suppliers.supplier_catalog
# Quarantine: silver.quarantine.src_09_suppliers
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_09_suppliers.supplier_catalog'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_09_suppliers'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_09_suppliers')
print('Silver Source 09 SFTP Suppliers — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_09_suppliers.supplier_catalog')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Cast all string numerics to correct types
df = bronze \
    .withColumn('unit_cost_gbp', F.expr('try_cast(unit_cost_gbp as double)')) \
    .withColumn('rrp_gbp',       F.expr('try_cast(rrp_gbp as double)')) \
    .withColumn('stock_qty',     F.expr('try_cast(stock_qty as long)')) \
    .withColumn('lead_days',     F.expr('try_cast(lead_days as integer)')) \
    .withColumn('updated_date',  F.to_date(F.col('updated_date')))

# Step 2: Normalise
df = df \
    .withColumn('product_sku',  F.upper(F.trim(F.col('product_sku')))) \
    .withColumn('warehouse',    F.upper(F.trim(F.col('warehouse')))) \
    .withColumn('file_format',  F.lower(F.trim(F.col('file_format'))))

# Step 3: Bad rows
bad = df.filter(
    F.col('product_sku').isNull() |
    F.col('unit_cost_gbp').isNull() |
    (F.col('unit_cost_gbp') <= 0) |
    (F.col('stock_qty') < 0)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('supplier_catalog'))

# Step 4: Good rows — dedup on product_sku keep latest
good = df.filter(
    F.col('product_sku').isNotNull() &
    F.col('unit_cost_gbp').isNotNull() &
    (F.col('unit_cost_gbp') > 0) &
    ((F.col('stock_qty') >= 0) | F.col('stock_qty').isNull())
)

# Dedup — keep row with latest updated_date, nulls last
w = Window.partitionBy('product_sku').orderBy(
    F.col('updated_date').desc_nulls_last()
)
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'Supplier catalog: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Validate cost < retail where both present
cost_violations = good.filter(
    F.col('rrp_gbp').isNotNull() &
    (F.col('unit_cost_gbp') >= F.col('rrp_gbp'))
).count()
print(f'Cost >= retail violations: {cost_violations} (should be 0)')

# Step 5: Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.product_sku = s.product_sku') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 6: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_09_suppliers').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')
else:
    print('No quarantine rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_09_suppliers.supplier_catalog: {count} rows')
spark.sql(f"""
    SELECT product_sku, unit_cost_gbp, rrp_gbp, stock_qty, lead_days
    FROM {TARGET_TABLE}
    LIMIT 5
""").show(truncate=False)
